### Try-It Activity 18.1: Comparing Methods


This Try-It activity focuses on weighing the positives and negatives of different estimators and vectorization strategies for a text classification problem.  In order to consider each of these components, you should make use of the `Pipeline` and `GridSearchCV` objects in Scikit-Learn to try different combinations of vectorizers with different estimators.  For each of these, you also want to use the `.cv_results_` to examine the time for the estimator to fit the data.

### The Data

The dataset below is from [kaggle]() and contains a dataset named the "ColBert Dataset" created for this [paper](https://arxiv.org/pdf/2004.12765.pdf).  You are to use the text column to classify whether or not the text was humorous.  It is loaded and displayed below.


In [1]:
%pip install nltk



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import time
import nltk

from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

df = pd.read_csv('data/dataset-minimal.csv')
print(df.shape)
print(df['humor'].value_counts())
df.head()

(99999, 2)
humor
False    50021
True     49978
Name: count, dtype: int64


,text,humor
0,"Joe biden rules out 2020 bid: 'guys, i'm not r...",False
1,Watch: darvish gave hitter whiplash with slow ...,False
2,What do you call a turtle without its shell? d...,True
3,5 reasons the 2016 election feels so personal,False
4,"Pasco police shot mexican migrant from behind,...",False


In [3]:
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

#### Task


**Text preprocessing:** As a pre-processing step, perform both `stemming` and `lemmatizing` to normalize your text before classifying. For each technique use both the `CountVectorize`r and `TfidifVectorizer` and use options for stop words and max features to prepare the text data for your estimator.

**Classification:** Once you have prepared the text data with stemming lemmatizing techniques, consider `LogisticRegression`, `DecisionTreeClassifier`, and `MultinomialNB` as classification algorithms for the data. Compare their performance in terms of accuracy and speed.

Share the results of your best classifier in the form of a table with the best version of each estimator, a dictionary of the best parameters and the best score.

In [4]:
stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def stem_text(text):
    return ' '.join([stemmer.stem(word) for word in text.split()])

def lemmatize_text(text):
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

# Apply both transformations
df['text_stemmed']    = df['text'].apply(stem_text)
df['text_lemmatized'] = df['text'].apply(lemmatize_text)

print("Original:   ", df['text'].iloc[2])
print("Stemmed:    ", df['text_stemmed'].iloc[2])
print("Lemmatized: ", df['text_lemmatized'].iloc[2])

Original:    What do you call a turtle without its shell? dead.
Stemmed:     what do you call a turtl without it shell? dead.
Lemmatized:  What do you call a turtle without it shell? dead.


In [5]:
# We will run experiments for each text version
X_orig  = df['text']
X_stem  = df['text_stemmed']
X_lemma = df['text_lemmatized']
y       = df['humor']

# Use lemmatized as primary (we'll compare all three)
X_train_o, X_test_o, y_train, y_test = train_test_split(
    X_orig, y, test_size=0.2, random_state=42)

X_train_s, X_test_s, _, _ = train_test_split(
    X_stem, y, test_size=0.2, random_state=42)

X_train_l, X_test_l, _, _ = train_test_split(
    X_lemma, y, test_size=0.2, random_state=42)

print(f"Train: {len(X_train_o):,} / Test: {len(X_test_o):,}")

Train: 79,999 / Test: 20,000


In [6]:
# Define models
models = {
    'Logistic':      LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Bayes':         MultinomialNB()
}

# Define vectorizers
vectorizers = {
    'CountVectorizer': CountVectorizer(),
    'TfidfVectorizer': TfidfVectorizer()
}

# Define text versions
text_versions = {
    'original':    (X_train_o, X_test_o),
    'stemmed':     (X_train_s, X_test_s),
    'lemmatized':  (X_train_l, X_test_l)
}

# Parameter grids per model
param_grids = {
    'Logistic': {
        'vectorizer__max_features': [5000, 10000],
        'vectorizer__stop_words': ['english', None],
        'classifier__C': [0.1, 1, 10]
    },
    'Decision Tree': {
        'vectorizer__max_features': [5000, 10000],
        'vectorizer__stop_words': ['english', None],
        'classifier__max_depth': [10, 20, None]
    },
    'Bayes': {
        'vectorizer__max_features': [5000, 10000],
        'vectorizer__stop_words': ['english', None],
        'classifier__alpha': [0.1, 0.5, 1.0]
    }
}

all_results = []

for text_name, (X_tr, X_te) in text_versions.items():
    for vec_name, vectorizer in vectorizers.items():
        for model_name, model in models.items():
            print(f"Running: {text_name} | {vec_name} | {model_name}...")
            
            pipe = Pipeline([
                ('vectorizer',  vectorizer),
                ('classifier',  model)
            ])
            
            grid = GridSearchCV(
                pipe,
                param_grid=param_grids[model_name],
                cv=5,
                scoring='accuracy',
                n_jobs=-1
            )
            
            start = time.time()
            grid.fit(X_tr, y_train)
            fit_time = round(time.time() - start, 2)
            
            all_results.append({
                'text':        text_name,
                'vectorizer':  vec_name,
                'model':       model_name,
                'best_score':  round(grid.best_score_, 4),
                'best_params': grid.best_params_,
                'fit_time':    fit_time
            })
            
            print(f"  Best score: {grid.best_score_:.4f} | Time: {fit_time}s")

print("\nDone!")


Running: original | CountVectorizer | Logistic...


  Best score: 0.9228 | Time: 69.09s
Running: original | CountVectorizer | Decision Tree...
  Best score: 0.8581 | Time: 190.95s
Running: original | CountVectorizer | Bayes...
  Best score: 0.9110 | Time: 21.77s
Running: original | TfidfVectorizer | Logistic...
  Best score: 0.9193 | Time: 24.58s
Running: original | TfidfVectorizer | Decision Tree...
  Best score: 0.8474 | Time: 188.61s
Running: original | TfidfVectorizer | Bayes...
  Best score: 0.9075 | Time: 22.19s
Running: stemmed | CountVectorizer | Logistic...
  Best score: 0.9206 | Time: 28.22s
Running: stemmed | CountVectorizer | Decision Tree...
  Best score: 0.8580 | Time: 127.84s
Running: stemmed | CountVectorizer | Bayes...
  Best score: 0.9103 | Time: 21.94s
Running: stemmed | TfidfVectorizer | Logistic...
  Best score: 0.9182 | Time: 24.39s
Running: stemmed | TfidfVectorizer | Decision Tree...
  Best score: 0.8483 | Time: 195.89s
Running: stemmed | TfidfVectorizer | Bayes...
  Best score: 0.9073 | Time: 21.79s
Running: lem

In [9]:
results_df = pd.DataFrame(all_results)

# Best result per model across all combinations
best_per_model = results_df.loc[
    results_df.groupby('model')['best_score'].idxmax()
][['model', 'text', 'vectorizer', 'best_score', 'best_params', 'fit_time']]

best_per_model = best_per_model.set_index('model')
best_per_model

,text,vectorizer,best_score,best_params,fit_time
model,,,,,
Bayes,lemmatized,CountVectorizer,0.9111,"{'classifier__alpha': 0.5, 'vectorizer__max_fe...",21.65
Decision Tree,original,CountVectorizer,0.8581,"{'classifier__max_depth': None, 'vectorizer__m...",190.95
Logistic,original,CountVectorizer,0.9228,"{'classifier__C': 1, 'vectorizer__max_features...",69.09


In [10]:
# Final table as requested in the notebook
final_table = pd.DataFrame({
    'model':       best_per_model.index,
    'best_params': best_per_model['best_params'].values,
    'best_score':  best_per_model['best_score'].values
}).set_index('model')

final_table

,best_params,best_score
model,,
Bayes,"{'classifier__alpha': 0.5, 'vectorizer__max_fe...",0.9111
Decision Tree,"{'classifier__max_depth': None, 'vectorizer__m...",0.8581
Logistic,"{'classifier__C': 1, 'vectorizer__max_features...",0.9228
